# Clase 36 - Notebook 2 - algoritmo de optimización NSGA-II
En este Notebook abordamos la simulación y optimización de este proceso mediante un modelo dinámico. El problema de optimización multiobjetivo busca minimizar simultáneamente el **contenido energético, $E$** de la corriente de aire caliente y la **humedad residual, $X_0$** del polvo.

$$\begin{aligned} & \text{Minimize} \quad \text{ContenidoEnergetico},\ X_0 \\[4pt] & \text{subject to} \\[2pt] & 462 \le G \le 858, \\[2pt] & 3.5 \cdot 10^{-5} \le r_d \le 6.5 \cdot 10^{-5} \end{aligned}$$


### ⚙️ Paso 0: Configuración Automática del Entorno (Google Colab)
Si estás ejecutando este cuaderno en **Google Colab**, ejecuta la siguiente celda una sola vez al inicio de la sesión para clonar automáticamente el repositorio e instalar todas las dependencias requeridas.


In [ ]:
# --- Preámbulo Universal para Google Colab y Entornos Locales ---
import os, sys

if 'google.colab' in sys.modules:
    REPO_DIR = '/content/DAII-SprayDrying'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/felipehuerta17/DAII-SprayDrying.git {REPO_DIR}
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    %pip install -q numpy scipy pandas matplotlib pymoo casadi
    print('✅ Entorno de Google Colab configurado con éxito.')
else:
    print('✅ Ejecutando en entorno local.')


# 02 — NSGA-II (pymoo)
Genera el frente y guarda CSV/figura en `./outputs`.


In [ ]:
import numpy as np, pandas as pd, os, time, matplotlib.pyplot as plt
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from spraydrylib.pymoo_problem import MOOSprayProblem

os.makedirs("./outputs", exist_ok=True)
problem = MOOSprayProblem(tf=400.0, n_steps=600)
algorithm = NSGA2(pop_size=100)
termination = get_termination("n_gen", 20)

t0 = time.perf_counter()
res = minimize(problem, algorithm, termination, seed=1, save_history=True, verbose=False)
elapsed = time.perf_counter() - t0
print(f"Tiempo aprox: {elapsed:.2f} s")

X = np.asarray(res.X, float); F = np.asarray(res.F, float)
stamp = time.strftime("%Y%m%d-%H%M%S")
csv = f'./outputs/front_nsga2_{stamp}.csv'
df = pd.DataFrame(np.hstack([X,F]), columns=["G_kg_h","rd_m","Energia_kW","Xo_prom_ultimos"])
df.to_csv(csv, index=False)
print("Guardado:", csv, "| puntos:", len(X))

plt.figure(figsize=(7.5,4.5))
plt.scatter(F[:,0], F[:,1], s=36, facecolor="#1f77b4", edgecolor="black", alpha=0.95, label="Frente de Pareto - NSGAII")
plt.xlabel("Gasto energético (kW)")
plt.ylabel(r"Contenido de agua en el droplet $\left(\frac{kg_{agua}}{kg_{sólidos}}\right)$")
plt.title("Frente NSGA-II")
plt.legend(); plt.tight_layout()
plt.savefig(f'./outputs/pareto_nsga2_{stamp}.png', dpi=200); plt.savefig(f'./outputs/pareto_nsga2_{stamp}.svg')
plt.show()


# Selección manual de soluciones óptimas


In [ ]:
df.sort_values(by="Energia_kW").head(5)


In [ ]:
df.sort_values(by="Xo_prom_ultimos").head(5)
